# Chapter 8: Observability and Gauge Freedom

<a href="../lite/lab/index.html?path=ch08_observability_gauge.ipynb" target="_blank" style="display:inline-block;padding:8px 18px;background:#1976d2;color:white;border-radius:5px;text-decoration:none;font-weight:bold;font-size:0.95em;">&#9654; Open in JupyterLite: run and edit this notebook</a>

*Runs entirely in your browser, no installation required.*

**How to use:** Edit the parameter values in each cell and re-run it to explore.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.linalg import null_space

%matplotlib inline
plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def draw_cov_ellipse(ax, mu, cov, n_std=2, **kwargs):
    """Draw a covariance ellipse at `n_std` standard deviations."""
    evals, evecs = np.linalg.eigh(cov)
    evals = np.maximum(evals, 1e-12)
    angle = np.degrees(np.arctan2(evecs[1, 1], evecs[0, 1]))
    w, h = 2 * n_std * np.sqrt(evals[1]), 2 * n_std * np.sqrt(evals[0])
    ell = patches.Ellipse(mu, w, h, angle=angle, fill=False, **kwargs)
    ax.add_patch(ell)
    return ell

print('Imports OK \u2713')

Imagine you wake up in a perfectly featureless white room. You can see the walls and measure distances to them. Can you figure out where you are on Earth? No. Can you figure out your position relative to the walls? Yes.

That gap between what you *can* and *cannot* determine is called **gauge freedom**, and it haunts every SLAM system. This chapter explains what is **observable**, what is not, and why that matters.

```{admonition} What you will build
:class: tip

- Compute the null space of a SLAM information matrix to find unobservable directions
- Show that absolute position cannot be determined from relative measurements alone
- Demonstrate scale ambiguity in monocular vision
- Fix gauge freedom by anchoring the first pose and make the system solvable

**Real world application:** Every SLAM system has gauge freedom. If you do not fix it, your optimizer will wander or crash. After this chapter, you will know exactly what to anchor and why.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **numpy.linalg.svd** | SVD for null space computation and rank analysis |
| **GTSAM** | Factor graph library that handles gauge freedom automatically via constrained optimization |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **numpy.linalg.svd** | SVD for null space computation and rank analysis |
| **GTSAM** | Factor graph library that handles gauge freedom automatically via constrained optimization |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

## 8.1 What Is Observable?

A state is **observable** if we can determine it from measurements alone. In robotics, **relative** positions between landmarks are often observable, but **absolute** position may not be.

Consider the simplest possible example: two landmarks $\ell_1$ and $\ell_2$ on a 1D line. A robot measures the distance between them:

$$z = \ell_2 - \ell_1$$

From this single measurement we can recover $\ell_2 - \ell_1$, but we cannot recover $\ell_1$ or $\ell_2$ individually. Every pair $(\ell_1, \ell_2)$ satisfying the constraint lies on the same line in state space. Let's see this.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
z_measured = 3.0    # measured distance l2 - l1   (try 1.0, 3.0, 5.0)
sigma_z    = 0.4    # measurement noise std dev   (try 0.1, 0.4, 1.0)
# ─────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(7, 7))

# All (l1, l2) pairs consistent with the measurement
l1_vals = np.linspace(-5, 5, 400)
l2_center = l1_vals + z_measured

# Shade the +/- 2 sigma band
ax.fill_between(l1_vals, l1_vals + z_measured - 2*sigma_z,
                l1_vals + z_measured + 2*sigma_z,
                color='steelblue', alpha=0.2, label=f'$\\pm 2\\sigma$ band')
ax.plot(l1_vals, l2_center, 'steelblue', lw=2.5,
        label=f'$\\ell_2 = \\ell_1 + {z_measured:.1f}$')

# Mark a few specific solutions
for l1_ex in [-2, 0, 2, 4]:
    ax.plot(l1_ex, l1_ex + z_measured, 'o', color='tomato', ms=8, zorder=5)
    ax.annotate(f'({l1_ex}, {l1_ex + z_measured:.0f})',
                (l1_ex, l1_ex + z_measured), textcoords='offset points',
                xytext=(8, -12), fontsize=9)

ax.set_xlabel('$\\ell_1$ (landmark 1 position)')
ax.set_ylabel('$\\ell_2$ (landmark 2 position)')
ax.set_title('All $(\\ell_1, \\ell_2)$ pairs consistent with measurement $z$')
ax.set_aspect('equal')
ax.legend(fontsize=10)
ax.set_xlim(-5, 5); ax.set_ylim(-3, 9)
plt.tight_layout(); plt.show()

print(f'Measurement z = {z_measured:.1f} constrains l2 - l1, '
      f'but any point on the blue line is equally valid.')
print(f'The LINE is the observable manifold; sliding along it is the gauge freedom.')

**Key insight:** the measurement constrains one *direction* in state space ($\ell_2 - \ell_1$) but leaves another direction ($\ell_1 + \ell_2$, the global shift) completely free. That free direction is the **unobservable subspace**.

## 8.2 What Is Not Observable?

**Absolute position**, **absolute orientation**, and (in monocular vision) **absolute scale** are common unobservable quantities in SLAM.

We can detect unobservability systematically using the **observation matrix** $H$ (the Jacobian of the measurement function with respect to the state). If the rank of $H$ is less than the state dimension, some directions in state space cannot be determined from measurements. The **null space** of $H$ tells us exactly which directions are invisible.

### Example: 1D SLAM with 3 poses and 2 landmarks

State vector: $\mathbf{x} = [p_1, p_2, p_3, \ell_1, \ell_2]^\top$ (3 poses + 2 landmarks, all in 1D).

Measurements: each pose observes each landmark via $z = \ell_j - p_i$. The Jacobian row for one such measurement is $\frac{\partial z}{\partial \mathbf{x}}$, with $-1$ at the pose entry and $+1$ at the landmark entry.

Let's build $H$, check its rank, and find the null space.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
n_poses     = 3   # number of robot poses      (try 2, 3, 5)
n_landmarks = 2   # number of landmarks         (try 1, 2, 4)
# ─────────────────────────────────────────────────────────────────────────────

state_dim = n_poses + n_landmarks

# Build H: one row per (pose, landmark) measurement pair
rows = []
labels = []
for i in range(n_poses):
    for j in range(n_landmarks):
        row = np.zeros(state_dim)
        row[i] = -1.0                   # d(z)/d(p_i)
        row[n_poses + j] = 1.0          # d(z)/d(l_j)
        rows.append(row)
        labels.append(f'p{i+1}->L{j+1}')

H = np.array(rows)
rank_H = np.linalg.matrix_rank(H)
ns = null_space(H)

print('Observation matrix H:')
state_names = [f'p{i+1}' for i in range(n_poses)] + [f'L{j+1}' for j in range(n_landmarks)]
header = '          ' + '  '.join(f'{s:>5s}' for s in state_names)
print(header)
for lbl, row in zip(labels, H):
    print(f'{lbl:>8s}  ' + '  '.join(f'{v:5.0f}' for v in row))

print(f'\nState dimension: {state_dim}')
print(f'Rank of H:       {rank_H}')
print(f'Null space dim:  {state_dim - rank_H}')

if ns.shape[1] > 0:
    print(f'\nNull space basis (columns):')
    for col_idx in range(ns.shape[1]):
        v = ns[:, col_idx]
        # Normalize so largest element = 1 for readability
        v = v / v[np.argmax(np.abs(v))]
        desc = ', '.join(f'{state_names[k]}={v[k]:+.2f}' for k in range(len(v)))
        print(f'  Direction {col_idx+1}: [{desc}]')
    print('\nInterpretation: shifting ALL poses and landmarks by the same amount')
    print('leaves every measurement unchanged. This is the global translation gauge.')
else:
    print('\nThe system is fully observable (no null space).')

The null space is $[1, 1, 1, 1, 1]^\top$, meaning a uniform shift of all positions is invisible to the measurements. This is **global translation ambiguity**: you can slide the entire map and trajectory together without changing any relative observation.

## 8.3 Scale Ambiguity

Monocular vision cannot determine **absolute scale**. If every 3D point is twice as far away and the camera baseline is twice as large, the projected images are identical.

Formally, for a pinhole camera with projection $\pi(\mathbf{X}) = \frac{1}{Z}[X, Y]^\top$, scaling the scene by $k$ gives:

$$\pi(k\mathbf{X}) = \frac{1}{kZ}[kX, kY]^\top = \frac{1}{Z}[X, Y]^\top = \pi(\mathbf{X})$$

Let's verify this numerically: generate a 3D scene, project it from two cameras, then scale everything and confirm the projections match.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
scale_factor = 2.0    # scene scale multiplier   (try 0.5, 2.0, 5.0, 10.0)
n_points     = 12     # number of 3D points      (try 8, 12, 20)
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)

# Generate 3D points in front of both cameras
pts_3d = np.random.randn(n_points, 3) * np.array([2, 2, 0.5]) + np.array([0, 0, 5])

# Two cameras: cam1 at origin, cam2 shifted along x by baseline b
baseline = 1.0
f = 1.0  # focal length (normalized)

def project(pts, cam_pos, f=1.0):
    """Simple pinhole projection (no rotation, camera looks along +z)."""
    rel = pts - cam_pos
    u = f * rel[:, 0] / rel[:, 2]
    v = f * rel[:, 1] / rel[:, 2]
    return np.column_stack([u, v])

# Original scene
cam1_pos = np.array([0, 0, 0])
cam2_pos = np.array([baseline, 0, 0])
proj1_orig = project(pts_3d, cam1_pos, f)
proj2_orig = project(pts_3d, cam2_pos, f)

# Scaled scene: scale both points AND baseline
pts_3d_scaled = pts_3d * scale_factor
cam1_scaled = cam1_pos * scale_factor
cam2_scaled = cam2_pos * scale_factor
proj1_scaled = project(pts_3d_scaled, cam1_scaled, f)
proj2_scaled = project(pts_3d_scaled, cam2_scaled, f)

# Check projections match
max_diff_1 = np.max(np.abs(proj1_orig - proj1_scaled))
max_diff_2 = np.max(np.abs(proj2_orig - proj2_scaled))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Left: 3D side view
ax = axes[0]
ax.scatter(pts_3d[:, 0], pts_3d[:, 2], s=50, c='steelblue', label=f'Scale 1', zorder=3)
ax.scatter(pts_3d_scaled[:, 0], pts_3d_scaled[:, 2], s=50, c='tomato',
           marker='^', label=f'Scale {scale_factor}', zorder=3)
ax.plot(*cam1_pos[[0,2]], 'ks', ms=10)
ax.plot(*cam2_pos[[0,2]], 'k^', ms=10)
ax.plot(*cam2_scaled[[0,2]], 'r^', ms=10)
ax.set_xlabel('X'); ax.set_ylabel('Z (depth)')
ax.set_title('3D scene (top view)')
ax.legend(fontsize=9)

# Middle: Camera 1 projections
ax = axes[1]
ax.scatter(proj1_orig[:, 0], proj1_orig[:, 1], s=60, c='steelblue',
           label='Scale 1', zorder=3)
ax.scatter(proj1_scaled[:, 0], proj1_scaled[:, 1], s=30, c='tomato',
           marker='x', linewidths=2, label=f'Scale {scale_factor}', zorder=4)
ax.set_xlabel('u'); ax.set_ylabel('v')
ax.set_title('Camera 1 projection')
ax.legend(fontsize=9)

# Right: Camera 2 projections
ax = axes[2]
ax.scatter(proj2_orig[:, 0], proj2_orig[:, 1], s=60, c='steelblue',
           label='Scale 1', zorder=3)
ax.scatter(proj2_scaled[:, 0], proj2_scaled[:, 1], s=30, c='tomato',
           marker='x', linewidths=2, label=f'Scale {scale_factor}', zorder=4)
ax.set_xlabel('u'); ax.set_ylabel('v')
ax.set_title('Camera 2 projection')
ax.legend(fontsize=9)

plt.tight_layout(); plt.show()

print(f'Max projection difference (cam 1): {max_diff_1:.2e}')
print(f'Max projection difference (cam 2): {max_diff_2:.2e}')
print(f'\nThe two scenes look completely different in 3D,')
print(f'but produce IDENTICAL images. Scale is unobservable from images alone.')

The projections overlap perfectly despite the scene being scaled by a factor of $k$. This is why **monocular visual SLAM** always needs an external scale reference (a known object size, an IMU, or wheel odometry) to pin down the true metric scale.

## 8.4 Global Frame Ambiguity

SLAM can only determine geometry **relative to some reference**. The choice of that reference is a **gauge**. Rotating and translating the entire solution (poses + landmarks together) leaves every measurement unchanged, so the cost function evaluates to the same value.

Let's build the same SLAM solution in two different global frames and confirm they are equivalent.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
rotation_deg = 35.0   # rotation of alternative frame   (try 0, 45, 90, 180)
tx           = 3.0    # x translation                   (try -5, 0, 3)
ty           = -2.0   # y translation                   (try -2, 0, 4)
# ─────────────────────────────────────────────────────────────────────────────

# "Ground truth" SLAM solution in frame A
poses_A = np.array([[0, 0], [1, 0.5], [2, 1.5], [3, 1.0], [4, 0.0]])
landmarks_A = np.array([[1.5, 2.5], [3.5, 2.0], [0.5, -1.0]])

# Transform to frame B
theta = np.radians(rotation_deg)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
t = np.array([tx, ty])

poses_B = (R @ poses_A.T).T + t
landmarks_B = (R @ landmarks_A.T).T + t

# Compute all relative measurements in both frames
cost_A = 0.0
cost_B = 0.0
for i in range(len(poses_A)):
    for j in range(len(landmarks_A)):
        diff_A = landmarks_A[j] - poses_A[i]
        diff_B = landmarks_B[j] - poses_B[i]
        # Relative distance (the measurement)
        dist_A = np.linalg.norm(diff_A)
        dist_B = np.linalg.norm(diff_B)
        cost_A += (dist_A - dist_A)**2   # zero by definition for frame A
        cost_B += (dist_A - dist_B)**2   # compare to frame A "measurements"

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, poses, lm, label in [
    (axes[0], poses_A, landmarks_A, 'Frame A (original)'),
    (axes[1], poses_B, landmarks_B, f'Frame B (rotated {rotation_deg:.0f}\u00b0, shifted [{tx},{ty}])')
]:
    # Draw observation lines
    for i in range(len(poses)):
        for j in range(len(lm)):
            ax.plot([poses[i, 0], lm[j, 0]], [poses[i, 1], lm[j, 1]],
                    'gray', lw=0.5, alpha=0.4)
    # Draw trajectory
    ax.plot(poses[:, 0], poses[:, 1], 'o-', color='steelblue', lw=2,
            ms=8, label='Poses', zorder=4)
    # Draw landmarks
    ax.scatter(lm[:, 0], lm[:, 1], s=120, c='tomato', marker='*',
               zorder=5, label='Landmarks')
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_aspect('equal')
    ax.legend(fontsize=9)

plt.tight_layout(); plt.show()

print(f'Cost difference between frames: {cost_B:.2e}')
print('Both solutions are equally valid; the measurements are identical.')
print('The choice of global frame is arbitrary: this is gauge freedom.')

The maps look different, but every distance between a pose and a landmark is preserved by the rigid transformation. SLAM only recovers the internal geometry; the global frame is your choice.

## 8.5 Identifiability: Fixing the Gauge

A system with gauge freedom has a **singular information matrix** $\Lambda = J^\top J$. Its determinant is (numerically) zero, and solving $\Lambda \mathbf{x} = \mathbf{b}$ is ill conditioned.

The fix is to **anchor** certain states, adding a strong prior that removes the ambiguity. Common strategies:

| Strategy | What it fixes |
|---|---|
| Fix first pose at origin | Global translation (and orientation if 2D/3D) |
| Known baseline distance | Scale |
| GPS measurement | Absolute position |

Let's build the information matrix for a small 1D SLAM problem, observe that it is singular, then fix it by anchoring pose 0.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
n_poses      = 4       # number of poses        (try 3, 4, 6)
n_landmarks  = 3       # number of landmarks    (try 2, 3, 5)
anchor_sigma = 0.001   # prior std on pose 0    (try 0.001, 0.01, 0.1, 10.0)
# ─────────────────────────────────────────────────────────────────────────────

state_dim = n_poses + n_landmarks

# True positions
np.random.seed(7)
true_poses = np.sort(np.random.uniform(0, 10, n_poses))
true_landmarks = np.sort(np.random.uniform(0, 10, n_landmarks))
x_true = np.concatenate([true_poses, true_landmarks])

# Build Jacobian: each pose observes each landmark (z = l_j - p_i)
rows_J = []
z_vec = []
sigma_obs = 0.5
for i in range(n_poses):
    for j in range(n_landmarks):
        row = np.zeros(state_dim)
        row[i] = -1.0 / sigma_obs
        row[n_poses + j] = 1.0 / sigma_obs
        rows_J.append(row)
        z_vec.append((true_landmarks[j] - true_poses[i]) / sigma_obs)

J = np.array(rows_J)
z = np.array(z_vec)

# Information matrix WITHOUT anchor
Lambda_no_anchor = J.T @ J
det_no_anchor = np.linalg.det(Lambda_no_anchor)
rank_no_anchor = np.linalg.matrix_rank(Lambda_no_anchor)

# Add anchor: prior on pose 0 (p0 = 0)
anchor_row = np.zeros(state_dim)
anchor_row[0] = 1.0 / anchor_sigma
J_anchored = np.vstack([J, anchor_row])
z_anchored = np.append(z, 0.0 / anchor_sigma)  # prior: p0 = 0

Lambda_anchored = J_anchored.T @ J_anchored
det_anchored = np.linalg.det(Lambda_anchored)
rank_anchored = np.linalg.matrix_rank(Lambda_anchored)

# Solve the anchored system
x_solved = np.linalg.solve(Lambda_anchored, J_anchored.T @ z_anchored)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Sparsity of information matrices
ax = axes[0]
ax.imshow(np.abs(Lambda_no_anchor) > 1e-10, cmap='Blues', aspect='equal')
ax.set_title(f'$\\Lambda$ without anchor\nrank={rank_no_anchor}, det={det_no_anchor:.1e}')
ax.set_xlabel('state index'); ax.set_ylabel('state index')

ax = axes[1]
ax.imshow(np.abs(Lambda_anchored) > 1e-10, cmap='Blues', aspect='equal')
ax.set_title(f'$\\Lambda$ with anchor\nrank={rank_anchored}, det={det_anchored:.1e}')
ax.set_xlabel('state index'); ax.set_ylabel('state index')

# Solved positions vs truth
ax = axes[2]
state_names = [f'p{i}' for i in range(n_poses)] + [f'L{j}' for j in range(n_landmarks)]
x_idx = np.arange(state_dim)
# Shift truth to same gauge (p0 = 0)
x_true_shifted = x_true - x_true[0]
ax.bar(x_idx - 0.15, x_true_shifted, 0.3, color='steelblue', label='Truth (shifted to p0=0)')
ax.bar(x_idx + 0.15, x_solved, 0.3, color='tomato', label='Solved (anchored)')
ax.set_xticks(x_idx); ax.set_xticklabels(state_names, fontsize=9)
ax.set_ylabel('Position'); ax.set_title('Recovered vs. true positions')
ax.legend(fontsize=9)

plt.tight_layout(); plt.show()

print(f'Without anchor: det(Lambda) = {det_no_anchor:.2e}  (singular!)')
print(f'With anchor:    det(Lambda) = {det_anchored:.2e}  (invertible)')
print(f'Max position error: {np.max(np.abs(x_solved - x_true_shifted)):.6f}')

Without the anchor, $\Lambda$ is singular and the system cannot be solved. Adding a single strong prior on pose 0 makes it full rank, and the solution matches truth exactly (up to the gauge choice $p_0 = 0$).

**Try it:** increase `anchor_sigma` to 10 or 100. The anchor becomes weak, and while the matrix is technically invertible, the solution becomes poorly conditioned.

## 8.6 Practical Implications

Different SLAM systems have different gauge freedoms, and therefore require different anchoring strategies.

| SLAM Flavor | Unobservable DoFs | Typical Gauge Fix |
|---|---|---|
| **2D LiDAR SLAM** | 2 translation + 1 rotation = 3 | Fix first pose at origin |
| **3D LiDAR SLAM** | 3 translation + 3 rotation = 6 | Fix first pose at origin |
| **Monocular Visual SLAM** | 3 trans + 3 rot + 1 scale = 7 | Fix first pose + set baseline = 1 |
| **Stereo Visual SLAM** | 3 translation + 3 rotation = 6 | Fix first pose (scale from stereo baseline) |
| **GPS aided SLAM** | 0 (GPS fixes position) | No anchor needed |

Let's see the difference anchoring makes on a small 2D SLAM problem.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
noise_sigma   = 0.05   # measurement noise       (try 0.01, 0.05, 0.2)
n_solve_tries = 6      # number of random inits  (try 3, 6, 10)
# ─────────────────────────────────────────────────────────────────────────────

# True 2D SLAM problem: 4 poses, 3 landmarks
true_poses_2d = np.array([[0, 0], [2, 0], [3, 2], [1, 3]])
true_lm_2d = np.array([[1, 1], [3, 1], [2, 3]])

n_p = len(true_poses_2d)
n_l = len(true_lm_2d)

# Generate noisy relative measurements (pose to landmark vectors)
np.random.seed(11)
measurements = []
for i in range(n_p):
    for j in range(n_l):
        diff = true_lm_2d[j] - true_poses_2d[i]
        z_noisy = diff + np.random.randn(2) * noise_sigma
        measurements.append((i, j, z_noisy))

def slam_cost(x_flat, measurements, n_p, n_l):
    """Sum of squared residuals for 2D pose-landmark SLAM."""
    poses = x_flat[:2*n_p].reshape(n_p, 2)
    lms = x_flat[2*n_p:].reshape(n_l, 2)
    cost = 0.0
    for i, j, z in measurements:
        pred = lms[j] - poses[i]
        residual = pred - z
        cost += np.sum(residual**2)
    return cost

from scipy.optimize import minimize

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for col, (do_anchor, title) in enumerate([
    (False, 'Without anchoring (gauge free)'),
    (True,  'With first pose anchored at origin')
]):
    ax = axes[col]
    colors_trial = plt.cm.Set2(np.linspace(0, 1, n_solve_tries))

    for trial in range(n_solve_tries):
        # Random initial guess
        np.random.seed(trial * 7 + 3)
        x0 = np.random.randn(2 * (n_p + n_l)) * 2

        if do_anchor:
            # Fix pose 0 at origin by adding a strong prior
            def cost_anchored(x_flat):
                c = slam_cost(x_flat, measurements, n_p, n_l)
                c += 1e6 * np.sum(x_flat[:2]**2)  # anchor pose 0
                return c
            res = minimize(cost_anchored, x0, method='L-BFGS-B')
        else:
            res = minimize(lambda x: slam_cost(x, measurements, n_p, n_l),
                          x0, method='L-BFGS-B')

        sol_poses = res.x[:2*n_p].reshape(n_p, 2)
        sol_lms = res.x[2*n_p:].reshape(n_l, 2)

        ax.plot(sol_poses[:, 0], sol_poses[:, 1], 'o-',
                color=colors_trial[trial], alpha=0.7, ms=5, lw=1.5)
        ax.scatter(sol_lms[:, 0], sol_lms[:, 1], s=40,
                   color=colors_trial[trial], marker='^', alpha=0.7)

    ax.set_title(title, fontsize=11)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_aspect('equal')

plt.tight_layout(); plt.show()

print('Left: each random initialization converges to a DIFFERENT global position,')
print('      because the gauge is free. All solutions have the same cost.')
print('Right: anchoring pose 0 fixes the gauge; all initializations converge')
print('       to the SAME solution.')

Without anchoring, the optimizer finds valid solutions scattered all over the plane. Each one is equally correct because the cost function has a continuous valley of minima (the gauge orbit). Anchoring the first pose eliminates that valley and produces a unique solution.

---

## Exercises

### Exercise 8.1: Rank and Null Space of an Observation Matrix

Given the observation matrix $H$ below (4 measurements, 5 states), compute its rank and null space. Interpret which state directions are unobservable.

In [ ]:
# Exercise 8.1
H_ex = np.array([
    [-1,  0,  0,  1,  0],   # pose 1 observes landmark 1
    [-1,  0,  0,  0,  1],   # pose 1 observes landmark 2
    [ 0, -1,  0,  1,  0],   # pose 2 observes landmark 1
    [ 0,  0, -1,  0,  1],   # pose 3 observes landmark 2
])

# TODO: compute rank of H_ex
rank = ...  # fill this in
print(f'Rank of H: {rank}')
print(f'State dimension: {H_ex.shape[1]}')
print(f'Unobservable directions: {H_ex.shape[1] - rank}')

# TODO: compute null space of H_ex
# ns = null_space(H_ex)
# print('Null space columns:')
# print(ns)

# TODO: interpret what each null space vector means physically

### Exercise 8.2: Triangle of Poses

Build the information matrix $\Lambda = J^\top J$ for 3 poses arranged in a triangle with relative measurements between consecutive poses: $z_{01} = p_1 - p_0$, $z_{12} = p_2 - p_1$, $z_{20} = p_0 - p_2$.

Find the null space. Then add one absolute measurement $z_{\text{abs}} = p_0$ and show the matrix becomes full rank.

In [ ]:
# Exercise 8.2
# State: x = [p0, p1, p2] in 1D

# TODO: Build J for the three relative measurements
# z01 = p1 - p0  =>  row = [-1, 1, 0]
# z12 = p2 - p1  =>  row = [0, -1, 1]
# z20 = p0 - p2  =>  row = [1, 0, -1]
J_triangle = np.array([
    # fill this in
])

# TODO: compute Lambda = J^T J
# Lambda = ...

# TODO: check rank, find null space

# TODO: add absolute measurement on p0 and verify full rank

### Exercise 8.3: Scale Ambiguity Numerically

Scale a set of 3D points and the stereo camera baseline by a factor $k$. Compute the reprojection errors in both cases and verify they are identical (within floating point tolerance).

In [ ]:
# Exercise 8.3
np.random.seed(99)
k = 3.0   # scale factor

# Generate 3D points
pts = np.random.randn(10, 3) * 2 + np.array([0, 0, 8])

# Camera positions
cam_left  = np.array([0, 0, 0])
cam_right = np.array([0.5, 0, 0])  # baseline = 0.5

# TODO: project points from both cameras
# TODO: scale points and baseline by k
# TODO: project again
# TODO: compare reprojection errors and print the max difference

### Exercise 8.4 (Challenge): Full 2D SLAM Gauge Analysis

Build a 2D SLAM problem with 5 poses and 3 landmarks. Each pose observes each landmark (relative position measurement in 2D). The state vector has $5 \times 2 + 3 \times 2 = 16$ elements.

1. Build the full Jacobian $J$ and information matrix $\Lambda = J^\top J$.
2. Visualize the sparsity pattern of $\Lambda$.
3. Find the null space dimensionality. (What do you expect for 2D translation + rotation?)
4. Fix the gauge by anchoring pose 0 (x and y) and one landmark's y coordinate (to fix rotation). Verify full rank.
5. Solve and plot the result.

In [ ]:
# Exercise 8.4 (Challenge)

# True positions
true_poses_ex = np.array([[0,0], [1,0], [2,1], [2,2], [1,2.5]])
true_lm_ex = np.array([[0.5, 1.5], [2.5, 0.5], [1.5, 3.0]])

n_p_ex = len(true_poses_ex)   # 5
n_l_ex = len(true_lm_ex)     # 3
state_dim_ex = 2 * (n_p_ex + n_l_ex)  # 16

# TODO: Build Jacobian J
# For measurement z_{ij} = l_j - p_i (2D vector):
#   d(z)/d(p_i) = -I (2x2),  d(z)/d(l_j) = +I (2x2)
# Each measurement contributes 2 rows to J

# TODO: Compute Lambda = J^T J
# TODO: Visualize sparsity with plt.spy() or imshow
# TODO: Find null space dimension
# TODO: Add anchoring priors and solve
# TODO: Plot recovered vs. true positions

---

**Chapter 8 Summary**

- **Observability** tells us which states we can determine from measurements; unobservable states live in the **null space** of the observation matrix.
- **Gauge freedom** is the set of transformations (translation, rotation, scale) that leave all measurements unchanged.
- **Monocular vision** has 7 degrees of gauge freedom (3 translation + 3 rotation + 1 scale).
- The **information matrix** $\Lambda = J^\top J$ is singular when gauge freedom exists.
- **Anchoring** (fixing a pose, setting a known scale) breaks the gauge and makes the system identifiable.
- Ignoring gauge freedom leads to singular matrices, numerical instability, and solutions that drift arbitrarily.